In [9]:
import pandas as pd

#Charger les données
df = pd.read_csv("../data/diabetes.csv")

# Séparer les données en ensemble d'entraînement (80%) et de test (20%)
# pour entraîner le modèle sur une partie et évaluer ses performances sur une autre
train = df.sample(frac=0.8, random_state=1)
# Création de l'ensemble de test en supprimant les lignes déjà utilisées dans l'entraînement
# permet d'évaluer le modèle sur des données qu'il n'a jamais vues
test = df.drop(train.index)

#Distance euclidienne
def distance(row1, row2):
    sum_val = 0
    for col in df.columns[:-1]:  # toutes les colonnes sauf Outcome (classe)
        # Calcul de la différence entre les valeurs des deux points pour une feature
        sum_val += (row1[col] - row2[col]) ** 2
    return sum_val ** 0.5

# Prédiction KNN
def predict(train, test_row, K):
    # Liste pour stocker les distances entre le point test et tous les points d'entraînement
    distances = []
    # Parcours de toutes les données d'entraînement
    for _, train_row in train.iterrows():
        # Calcul de la distance entre le point test et un point d'entraînement
        d = distance(test_row, train_row)
        # Stockage de la distance et de la classe associée
        distances.append((d, train_row["Outcome"]))

    # on trie selon la distance 
    distances.sort(key=lambda x: x[0])

    # Prendre les K voisins(selon K qu'on va choisi)
    neighbors = distances[:K]

    # Dictionnaire pour compter le nombre de votes de chaque classe
    votes = {}
    # Parcours des K voisins les plus proches
    for d, label in neighbors:
        # Si la classe existe déjà dans le dictionnaire, on ajoute 1
        # sinon on initialise à 0 puis on ajoute 1
        votes[label] = votes.get(label, 0) + 1

    # Retourner la classe majoritaire
    return max(votes, key=votes.get)

# Test du modèle
K = 5
correct = 0
# Parcours de chaque ligne du dataset de test
for _, row in test.iterrows():
    # Prédiction de la classe pour le point courant
    pred = predict(train, row, K)
    # Vérification si la prédiction est correcte
    if pred == row["Outcome"]:
        correct += 1
# Calcul de l'accuracy (taux de bonnes prédictions)
accuracy = correct / len(test)
print("Accuracy:", accuracy)


Accuracy: 0.7077922077922078
